# 5th : KNN

2026.04.29

In [4]:
import pickle
import numpy as np
import tarfile
import os

In [ ]:
data_tar_path = 'cifar-10-python.tar.gz'

with tarfile.open(data_tar_path, 'r:gz') as tar:
    tar.extractall('.')

In [5]:
def load_cifar10_data(file_path):
    with open(file_path, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    images = batch[b'data']
    labels = batch[b'labels']
    return images, np.array(labels)

data_dir = 'cifar-10-batches-py'

all_train_images = []
all_train_labels = []

for i in range(1, 6):  # data_batch_1 ~ data_batch_5
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    images, labels = load_cifar10_data(batch_path)
    all_train_images.append(images)
    all_train_labels.append(labels)

# load whole train dataset
all_train_images = np.concatenate(all_train_images, axis=0)
all_train_labels = np.concatenate(all_train_labels, axis=0)

# load whole test dataset
test_path = os.path.join(data_dir, 'test_batch')
all_test_images, all_test_labels = load_cifar10_data(test_path)

print(f"Train dataset: {all_train_images.shape}")
print(f"Test dataset: {all_test_images.shape}")

Train dataset: (50000, 3072)
Test dataset: (10000, 3072)


In [6]:
# Each label should have an equal number of samples for balanced training
# make subset (train : 500/class, test : 100/class)
TRAIN_PER_CLASS = 500
TEST_PER_CLASS  = 100

rng = np.random.default_rng(seed=42)

def stratified_sample(images, labels, n_per_class):
    selected_idx = []
    for class_id in range(10):  # 10 classes
        class_idx = np.where(labels == class_id)[0]
        chosen = rng.choice(class_idx, size=n_per_class, replace=False)
        selected_idx.append(chosen) # mixup
    
    selected_idx = np.concatenate(selected_idx)
    rng.shuffle(selected_idx)
    
    return images[selected_idx], labels[selected_idx]

train_images, train_labels = stratified_sample(all_train_images, all_train_labels, TRAIN_PER_CLASS)
test_images,  test_labels  = stratified_sample(all_test_images,  all_test_labels,  TEST_PER_CLASS)

print(f"train image shape: {train_images.shape}, train label: {train_labels.shape}")
print(f"test image shape:  {test_images.shape}, test label: {test_labels.shape}")

print("\n[Train] number of samples per class:")
for c in range(10):
    print(f"class {c}: {(train_labels == c).sum()}")

print("\n[Test] number of samples per class:")
for c in range(10):
    print(f"class {c}: {(test_labels == c).sum()}")

train image shape: (5000, 3072), train label: (5000,)
test image shape:  (1000, 3072), test label: (1000,)

[Train] number of samples per class:
class 0: 500
class 1: 500
class 2: 500
class 3: 500
class 4: 500
class 5: 500
class 6: 500
class 7: 500
class 8: 500
class 9: 500

[Test] number of samples per class:
class 0: 100
class 1: 100
class 2: 100
class 3: 100
class 4: 100
class 5: 100
class 6: 100
class 7: 100
class 8: 100
class 9: 100


output <br>
train image shape: (5000, 3072), train label: (5000,)  
test image shape:  (1000, 3072), test label: (1000,)  
already flatten

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
K_VALUES  = [1, 3, 5, 7, 9]
METRICS   = ['l1', 'l2']  # manhattan, euclidean
N_FOLDS   = 5
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# ── 4. 5-Fold Cross-Validation ─────────────────────────────────────────────────
print("── 5-Fold Cross-Validation ──")
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

mean_cv = {metric: {} for metric in METRICS}

for metric in METRICS:
    sklearn_metric = 'manhattan' if metric == 'l1' else 'euclidean'
    print(f"\nMetric: {metric.upper()}")
    for k in K_VALUES:
        knn  = KNeighborsClassifier(n_neighbors=k, metric=sklearn_metric, n_jobs=-1)
        scores = cross_val_score(knn, train_images, train_labels, cv=kf, scoring='accuracy')
        mean_cv[metric][k] = scores.mean()
        print(f"  K={k} | folds={np.round(scores, 4)} | mean={scores.mean():.4f}")

# Best K per metric
best_k = {metric: max(mean_cv[metric], key=mean_cv[metric].get) for metric in METRICS}
print(f"\nBest K → L1: {best_k['l1']}, L2: {best_k['l2']}")

# ── 5. Plot CV Accuracy vs K ───────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
for metric in METRICS:
    accs = [mean_cv[metric][k] for k in K_VALUES]
    plt.plot(K_VALUES, accs, marker='o', label=f'{metric.upper()} distance')

plt.xlabel('K value')
plt.ylabel('Mean Cross-Validation Accuracy')
plt.title('5-Fold CV Accuracy vs K (L1 & L2)')
plt.xticks(K_VALUES)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('cv_accuracy_vs_k.png', dpi=150)
plt.show()

# ── 6. Final Evaluation on Test Set ───────────────────────────────────────────
print("\n── Test Set Evaluation ──")
for metric in METRICS:
    sklearn_metric = 'manhattan' if metric == 'l1' else 'euclidean'
    k   = best_k[metric]
    knn = KNeighborsClassifier(n_neighbors=k, metric=sklearn_metric, n_jobs=-1)
    knn.fit(train_images, train_labels)
    preds = knn.predict(test_images)
    acc   = accuracy_score(test_labels, preds)
    print(f"\n{metric.upper()} | Best K={k} | Test Accuracy: {acc:.4f}")

    # Confusion matrix
    cm = confusion_matrix(test_labels, preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix — {metric.upper()} | K={k} | Acc={acc:.4f}')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{metric}.png', dpi=150)
    plt.show()

── 5-Fold Cross-Validation ──

Metric: L1
  K=1 | folds=[0.306 0.282 0.272 0.283 0.282] | mean=0.2850
  K=3 | folds=[0.292 0.274 0.272 0.274 0.265] | mean=0.2754
  K=5 | folds=[0.301 0.278 0.27  0.287 0.285] | mean=0.2842


In [ ]:
# test